In [2]:
# ============================================================
# 01_transaction_logic_check.ipynb
# Mục tiêu: Validate business logic của Transaction cluster
# Focus: orders, order_items, payments, shipments, returns, reviews
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

INPUT_DIR = Path('input')
COLOR_OK   = '#1D9E75'
COLOR_WARN = '#EF9F27'
COLOR_BAD  = '#E24B4A'

FILE_CONFIGS = {
    'products'    : {'date_cols': []},
    'customers'   : {'date_cols': ['signup_date']},
    'geography'   : {'date_cols': []},
    'orders'      : {'date_cols': ['order_date']},
    'order_items' : {'date_cols': []},
    'payments'    : {'date_cols': []},
    'shipments'   : {'date_cols': ['ship_date', 'delivery_date']},
    'returns'     : {'date_cols': ['return_date']},
    'reviews'     : {'date_cols': ['review_date']},
    'sales'       : {'date_cols': ['Date']},
}

dfs = {}
for name, cfg in FILE_CONFIGS.items():
    dfs[name] = pd.read_csv(INPUT_DIR / f'{name}.csv',
                            parse_dates=cfg['date_cols'] or False,
                            low_memory=False)
    print(f"  ✅ {name:<18} {dfs[name].shape[0]:>8,} rows")

  ✅ products              2,412 rows
  ✅ customers           121,930 rows
  ✅ geography            39,948 rows
  ✅ orders              646,945 rows
  ✅ order_items         714,669 rows
  ✅ payments            646,945 rows
  ✅ shipments           566,067 rows
  ✅ returns              39,939 rows
  ✅ reviews             113,551 rows
  ✅ sales                 3,833 rows


In [3]:
# ── CELL A: Revenue Definition — Trap #1 ────────────────────
# Câu hỏi cốt lõi: unit_price trong order_items là giá GỐC hay giá ĐÃ GIẢM?
# Nếu unit_price là giá sau giảm → revenue = unit_price * quantity
# Nếu unit_price là giá gốc → revenue = unit_price * quantity - discount_amount
# Cần reconcile với sales.csv và payments.csv

print("=" * 65)
print("💡 TRAP #1 — Revenue Definition: unit_price là giá gì?")
print("=" * 65)

oi = dfs['order_items'].copy()
pay = dfs['payments'].copy()
orders = dfs['orders'].copy()
sales = dfs['sales'].copy()

# Tính revenue theo 2 cách rồi so sánh với payments.payment_value
oi['rev_A'] = oi['unit_price'] * oi['quantity']                         # nếu unit_price đã net
oi['rev_B'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount'] # nếu unit_price là gross

rev_by_order = oi.groupby('order_id').agg(
    rev_A=('rev_A', 'sum'),
    rev_B=('rev_B', 'sum')
).reset_index()

merged = rev_by_order.merge(pay[['order_id','payment_value']], on='order_id')

# Tính % match (trong sai số 1 đồng)
match_A = (np.abs(merged['rev_A'] - merged['payment_value']) < 1).mean()
match_B = (np.abs(merged['rev_B'] - merged['payment_value']) < 1).mean()

print(f"\n  Method A: revenue = unit_price × qty")
print(f"    → Khớp payment_value: {match_A*100:.1f}%")
print(f"\n  Method B: revenue = unit_price × qty − discount_amount")
print(f"    → Khớp payment_value: {match_B*100:.1f}%")

# Xem sample diff
merged['diff_A'] = merged['rev_A'] - merged['payment_value']
merged['diff_B'] = merged['rev_B'] - merged['payment_value']
print(f"\n  Diff A (rev_A - payment_value) stats:")
print(merged['diff_A'].describe().to_string())
print(f"\n  Diff B (rev_B - payment_value) stats:")
print(merged['diff_B'].describe().to_string())

💡 TRAP #1 — Revenue Definition: unit_price là giá gì?

  Method A: revenue = unit_price × qty
    → Khớp payment_value: 61.6%

  Method B: revenue = unit_price × qty − discount_amount
    → Khớp payment_value: 100.0%

  Diff A (rev_A - payment_value) stats:
count   646945.0000
mean      1158.6879
std       2429.0943
min         -0.0000
25%          0.0000
50%          0.0000
75%       1180.8100
max      37183.1600

  Diff B (rev_B - payment_value) stats:
count   646945.0000
mean         0.0000
std          0.0000
min         -0.0000
25%          0.0000
50%          0.0000
75%          0.0000
max          0.0000


In [4]:
# ── CELL B: Revenue Reconciliation vs sales.csv ─────────────
# Đây là câu hỏi quan trọng nhất cho forecasting
# sales.csv có phải aggregation của order_items không?
print("\n" + "=" * 65)
print("💡 TRAP #2 — sales.csv vs order_items aggregate")
print("=" * 65)

# Chỉ tính từ orders có order_status != 'cancelled'
valid_statuses = orders[orders['order_status'] != 'cancelled'][['order_id','order_date']]

oi_valid = oi.merge(valid_statuses, on='order_id')
oi_valid['order_date'] = pd.to_datetime(oi_valid['order_date'])

# Tính daily revenue từ order_items (thử cả 2 method)
daily_A = (oi_valid.groupby('order_date')['rev_A'].sum()
           .reset_index().rename(columns={'rev_A':'rev_oi_A', 'order_date':'Date'}))
daily_B = (oi_valid.groupby('order_date')['rev_B'].sum()
           .reset_index().rename(columns={'rev_B':'rev_oi_B', 'order_date':'Date'}))

sales_cmp = (sales.rename(columns={'Revenue':'rev_sales'})
             .merge(daily_A, on='Date', how='left')
             .merge(daily_B, on='Date', how='left'))

# Correlation check
corr_A = sales_cmp['rev_sales'].corr(sales_cmp['rev_oi_A'])
corr_B = sales_cmp['rev_sales'].corr(sales_cmp['rev_oi_B'])
ratio_A = (sales_cmp['rev_oi_A'] / sales_cmp['rev_sales']).median()
ratio_B = (sales_cmp['rev_oi_B'] / sales_cmp['rev_sales']).median()

print(f"\n  Correlation sales.Revenue vs Method A : {corr_A:.4f}")
print(f"  Median ratio (oi_A / sales)           : {ratio_A:.4f}")
print(f"\n  Correlation sales.Revenue vs Method B : {corr_B:.4f}")
print(f"  Median ratio (oi_B / sales)           : {ratio_B:.4f}")

# Nếu ratio ~ 1 và corr ~ 1 → hai source nhất quán
# Nếu corr ~ 1 nhưng ratio != 1 → sales.csv đã được scale
print(f"\n  Sample comparison (10 ngày đầu):")
print(sales_cmp[['Date','rev_sales','rev_oi_A','rev_oi_B']].head(10).to_string(index=False))


💡 TRAP #2 — sales.csv vs order_items aggregate

  Correlation sales.Revenue vs Method A : 0.9985
  Median ratio (oi_A / sales)           : 0.9104

  Correlation sales.Revenue vs Method B : 0.9906
  Median ratio (oi_B / sales)           : 0.8842

  Sample comparison (10 ngày đầu):
      Date    rev_sales     rev_oi_A     rev_oi_B
2012-07-04 5123547.9400 4968762.1200 4968762.1200
2012-07-05 2751773.4500 2520721.7900 2520721.7900
2012-07-06 3054029.4200 2646490.5500 2646490.5500
2012-07-07 2667930.9400 2337896.9600 2337896.9600
2012-07-08 2360851.9000 2199273.3700 2199273.3700
2012-07-09 3548386.4600 2909927.1800 2909927.1800
2012-07-10 5234938.6200 4809139.3800 4809139.3800
2012-07-11 5582884.7800 5279686.0500 5279686.0500
2012-07-12 5734632.0200 5418901.4000 5418901.4000
2012-07-13 5309511.7100 4767890.6400 4767890.6400


In [5]:
# ── CELL C: order_status distribution & cancelled logic ─────
print("\n" + "=" * 65)
print("📋 ORDER STATUS — Distribution & Consistency")
print("=" * 65)

orders = dfs['orders'].copy()

print("  order_status distribution:")
print(orders['order_status'].value_counts().to_string())
print(f"\n  Total orders: {len(orders):,}")

# Shipments chỉ tồn tại cho shipped/delivered/returned
ship = dfs['shipments'].copy()
expected_ship_statuses = {'shipped', 'delivered', 'returned'}
orders_with_ship = orders[orders['order_id'].isin(ship['order_id'])]['order_status'].value_counts()
print(f"\n  order_status của các orders CÓ shipment record:")
print(orders_with_ship.to_string())

# Orders returned có trong returns table không?
ret = dfs['returns'].copy()
returned_orders = set(orders[orders['order_status'] == 'returned']['order_id'])
ret_orders_in_returns = set(ret['order_id'].unique())
in_both = returned_orders & ret_orders_in_returns
only_status = returned_orders - ret_orders_in_returns
only_returns_table = ret_orders_in_returns - returned_orders

print(f"\n  Orders với status='returned'         : {len(returned_orders):,}")
print(f"  Orders có record trong returns table  : {len(ret_orders_in_returns):,}")
print(f"  Có trong cả hai                       : {len(in_both):,}")
print(f"  ⚠️  Status=returned nhưng KHÔNG có returns record: {len(only_status):,}")
print(f"  ⚠️  Có returns record nhưng status != returned   : {len(only_returns_table):,}")


📋 ORDER STATUS — Distribution & Consistency
  order_status distribution:
order_status
delivered    516716
cancelled     59462
returned      36142
shipped       13773
paid          13577
created        7275

  Total orders: 646,945

  order_status của các orders CÓ shipment record:
order_status
delivered    516192
returned      36113
shipped       13762

  Orders với status='returned'         : 36,142
  Orders có record trong returns table  : 36,062
  Có trong cả hai                       : 36,062
  ⚠️  Status=returned nhưng KHÔNG có returns record: 80
  ⚠️  Có returns record nhưng status != returned   : 0


In [6]:
# ── CELL D: Shipping timeline logic ─────────────────────────
print("\n" + "=" * 65)
print("🚚 SHIPMENTS — Timeline Sanity")
print("=" * 65)

ship = dfs['shipments'].copy()
orders_date = dfs['orders'][['order_id','order_date']].copy()
orders_date['order_date'] = pd.to_datetime(orders_date['order_date'])

ship_merged = ship.merge(orders_date, on='order_id')
ship_merged['days_to_ship']    = (ship_merged['ship_date']     - ship_merged['order_date']).dt.days
ship_merged['days_to_deliver'] = (ship_merged['delivery_date'] - ship_merged['order_date']).dt.days
ship_merged['transit_days']    = (ship_merged['delivery_date'] - ship_merged['ship_date']).dt.days

print("  days_to_ship (order → ship):")
print(ship_merged['days_to_ship'].describe().to_string())
neg_ship = (ship_merged['days_to_ship'] < 0).sum()
print(f"  ⚠️  Ship TRƯỚC order_date: {neg_ship:,}")

print(f"\n  transit_days (ship → delivery):")
print(ship_merged['transit_days'].describe().to_string())
neg_transit = (ship_merged['transit_days'] < 0).sum()
print(f"  ⚠️  Delivery TRƯỚC ship_date: {neg_transit:,}")

print(f"\n  days_to_deliver (order → delivery):")
print(ship_merged['days_to_deliver'].describe().to_string())


🚚 SHIPMENTS — Timeline Sanity
  days_to_ship (order → ship):
count   566067.0000
mean         1.4984
std          1.1184
min          0.0000
25%          0.0000
50%          1.0000
75%          2.0000
max          3.0000
  ⚠️  Ship TRƯỚC order_date: 0

  transit_days (ship → delivery):
count   566067.0000
mean         4.4992
std          1.7070
min          2.0000
25%          3.0000
50%          4.0000
75%          6.0000
max          7.0000
  ⚠️  Delivery TRƯỚC ship_date: 0

  days_to_deliver (order → delivery):
count   566067.0000
mean         5.9976
std          2.0405
min          2.0000
25%          4.0000
50%          6.0000
75%          7.0000
max         10.0000


In [8]:
# ── CELL E: Returns logic & refund consistency ───────────────
print("\n" + "=" * 65)
print("↩️  RETURNS — Logic & Refund Sanity")
print("=" * 65)

ret = dfs['returns'].copy()
ret['return_date'] = pd.to_datetime(ret['return_date'])
oi = dfs['order_items'].copy()

# return_quantity <= quantity trong order_items không?
ret_check = ret.merge(oi[['order_id','product_id','quantity','unit_price','discount_amount']],
                      on=['order_id','product_id'], how='left')
ret_check['qty_violated'] = ret_check['return_quantity'] > ret_check['quantity']
print(f"  return_quantity > order quantity: {ret_check['qty_violated'].sum():,} rows")

# refund_amount hợp lý không? (so với unit_price * return_quantity)
ret_check['expected_refund_A'] = ret_check['return_quantity'] * ret_check['unit_price']
ret_check['expected_refund_B'] = (ret_check['return_quantity'] * ret_check['unit_price']
                                   - ret_check['discount_amount'] * ret_check['return_quantity'] / ret_check['quantity'])
ret_check['refund_diff_A'] = ret_check['refund_amount'] - ret_check['expected_refund_A']
ret_check['refund_diff_B'] = ret_check['refund_amount'] - ret_check['expected_refund_B']

print(f"\n  refund vs expected (unit_price × return_qty) diff:")
print(ret_check['refund_diff_A'].describe().to_string())

print(f"\n  Return reasons distribution:")
print(ret['return_reason'].value_counts().to_string())

print(f"\n  Return rate by category (join với products):")
prod = dfs['products'].copy()
ret_with_prod = ret.merge(prod[['product_id','category','size']], on='product_id', how='left')
oi_with_prod  = oi.merge(prod[['product_id','category','size']], on='product_id', how='left')

ret_by_cat = ret_with_prod.groupby('category')['return_id'].count()
oi_by_cat  = oi_with_prod.groupby('category')['order_id'].count()
return_rate = (ret_by_cat / oi_by_cat * 100).sort_values(ascending=False)
print(return_rate.to_string())


↩️  RETURNS — Logic & Refund Sanity
  return_quantity > order quantity: 1 rows

  refund vs expected (unit_price × return_qty) diff:
count    39943.0000
mean     -1041.4743
std       1467.4785
min     -19971.5500
25%      -1266.8150
50%       -509.2600
75%       -182.9700
max         -0.0500

  Return reasons distribution:
return_reason
wrong_size          13967
defective            8020
not_as_described     7035
changed_mind         6931
late_delivery        3986

  Return rate by category (join với products):
category
GenZ         5.7214
Outdoor      5.6618
Streetwear   5.5393
Casual       5.3937


In [9]:
# ── CELL F: Reviews — Rating distribution & timing ──────────
print("\n" + "=" * 65)
print("⭐ REVIEWS — Rating Distribution & Timing")
print("=" * 65)

rev = dfs['reviews'].copy()
rev['review_date'] = pd.to_datetime(rev['review_date'])
ship = dfs['shipments'].copy()

print(f"  Rating distribution:")
print(rev['rating'].value_counts().sort_index().to_string())
print(f"\n  Review rate: {len(rev):,} / {len(dfs['orders']):,} orders = {len(rev)/len(dfs['orders'])*100:.1f}%")

# Reviews chỉ nên đến từ 'delivered' orders
orders_status = dfs['orders'][['order_id','order_status']]
rev_with_status = rev.merge(orders_status, on='order_id', how='left')
print(f"\n  order_status của orders có review:")
print(rev_with_status['order_status'].value_counts().to_string())

# review_date có sau delivery_date không?
rev_ship = rev.merge(ship[['order_id','delivery_date']], on='order_id', how='left')
rev_ship['days_after_delivery'] = (rev_ship['review_date'] - rev_ship['delivery_date']).dt.days
early_reviews = (rev_ship['days_after_delivery'] < 0).sum()
print(f"\n  ⚠️  Reviews TRƯỚC delivery_date: {early_reviews:,}")
print(f"\n  Days after delivery (review timing):")
print(rev_ship['days_after_delivery'].describe().to_string())


⭐ REVIEWS — Rating Distribution & Timing
  Rating distribution:
rating
1     5772
2     9095
3    17016
4    36412
5    45256

  Review rate: 113,551 / 646,945 orders = 17.6%

  order_status của orders có review:
order_status
delivered    113551

  ⚠️  Reviews TRƯỚC delivery_date: 0

  Days after delivery (review timing):
count   113551.0000
mean        15.4923
std          8.6681
min          1.0000
25%          8.0000
50%         16.0000
75%         23.0000
max         30.0000


In [10]:
# ── CELL G: MCQ Answer Keys — Direct computation ────────────
# Dùng để trả lời chính xác 10 câu trắc nghiệm
print("\n" + "=" * 65)
print("🎯 MCQ ANSWER KEYS — Direct Computation")
print("=" * 65)

orders = dfs['orders'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Q1: Median inter-order gap (customers có >1 đơn)
multi_buyers = (orders.groupby('customer_id')
                .filter(lambda x: len(x) > 1)
                .sort_values(['customer_id','order_date']))
gaps = (multi_buyers.groupby('customer_id')['order_date']
        .apply(lambda x: x.diff().dt.days.dropna())
        .reset_index(drop=True))
q1 = gaps.median()
print(f"\n  Q1 — Median inter-order gap: {q1:.1f} ngày")

# Q2: Segment có gross margin cao nhất
prod = dfs['products'].copy()
prod['gm'] = (prod['price'] - prod['cogs']) / prod['price']
q2 = prod.groupby('segment')['gm'].mean().idxmax()
print(f"  Q2 — Segment highest gross margin: {q2}")

# Q3: Most common return reason for Streetwear
ret = dfs['returns'].copy()
ret_sw = ret.merge(prod[['product_id','category']], on='product_id')
q3 = ret_sw[ret_sw['category'] == 'Streetwear']['return_reason'].value_counts().idxmax()
print(f"  Q3 — Top return reason (Streetwear): {q3}")

# Q4: Traffic source với bounce_rate thấp nhất
wt = dfs['web_traffic'].copy() if 'web_traffic' in dfs else None
if wt is not None:
    q4 = wt.groupby('traffic_source')['bounce_rate'].mean().idxmin()
    print(f"  Q4 — Lowest bounce_rate source: {q4}")

# Q5: % order_items có promo_id not null
oi = dfs['order_items'].copy()
q5 = (oi['promo_id'].notna().mean() * 100)
print(f"  Q5 — % order_items with promo: {q5:.1f}%")

# Q6: Age group có average orders/customer cao nhất
cust = dfs['customers'].copy()
orders_per_cust = orders.groupby('customer_id')['order_id'].count().reset_index()
orders_per_cust.columns = ['customer_id', 'n_orders']
cust_orders = cust.merge(orders_per_cust, on='customer_id', how='left')
cust_orders['n_orders'] = cust_orders['n_orders'].fillna(0)
q6 = cust_orders[cust_orders['age_group'].notna()].groupby('age_group')['n_orders'].mean().idxmax()
print(f"  Q6 — Age group highest avg orders: {q6}")

# Q7: Region với highest total revenue — cần join geography
# (sales.csv không có geography, cần đi qua orders → geography)
geo = dfs['geography'].copy()
oi_orders = oi.merge(orders[['order_id','order_date','zip']], on='order_id')
oi_orders['rev'] = oi_orders['unit_price'] * oi_orders['quantity']
oi_geo = oi_orders.merge(geo[['zip','region']], on='zip', how='left')
q7 = oi_geo.groupby('region')['rev'].sum().idxmax()
q7_vals = oi_geo.groupby('region')['rev'].sum().sort_values(ascending=False)
print(f"  Q7 — Region highest revenue: {q7}")
print(f"       Detail: {q7_vals.to_dict()}")

# Q8: Payment method phổ biến nhất trong cancelled orders
cancelled_orders = orders[orders['order_status'] == 'cancelled']['order_id']
pay = dfs['payments'].copy()
q8 = pay[pay['order_id'].isin(cancelled_orders)]['payment_method'].value_counts().idxmax()
print(f"  Q8 — Top payment method (cancelled): {q8}")

# Q9: Size có return rate cao nhất
oi_size = oi.merge(prod[['product_id','size']], on='product_id')
ret_size = ret.merge(prod[['product_id','size']], on='product_id')
ret_by_size = ret_size.groupby('size')['return_id'].count()
oi_by_size  = oi_size.groupby('size')['order_id'].count()
q9 = (ret_by_size / oi_by_size).idxmax()
print(f"  Q9 — Size highest return rate: {q9}")
print(f"       Detail: {(ret_by_size / oi_by_size).sort_values(ascending=False).to_dict()}")

# Q10: Installment plan với avg payment_value cao nhất
q10 = pay.groupby('installments')['payment_value'].mean().idxmax()
print(f"  Q10 — Installment plan highest avg payment: {q10} kỳ")
print(f"        Detail: {pay.groupby('installments')['payment_value'].mean().sort_values(ascending=False).to_dict()}")


🎯 MCQ ANSWER KEYS — Direct Computation

  Q1 — Median inter-order gap: 144.0 ngày
  Q2 — Segment highest gross margin: Standard
  Q3 — Top return reason (Streetwear): wrong_size
  Q5 — % order_items with promo: 38.7%
  Q6 — Age group highest avg orders: 55+
  Q7 — Region highest revenue: East
       Detail: {'East': 7637532676.2, 'Central': 4941908471.68, 'West': 3851035437.65}
  Q8 — Top payment method (cancelled): credit_card
  Q9 — Size highest return rate: S
       Detail: {'S': 0.05651526952720847, 'L': 0.05624978345479113, 'M': 0.05566009930396536, 'XL': 0.05520010361352157}
  Q10 — Installment plan highest avg payment: 6 kỳ
        Detail: {6: 24446.65440296606, 3: 24399.63548611777, 12: 24245.77269408417, 1: 24113.274165734634, 2: 708.4737294332724}


In [24]:
# ── CELL H: Sales Revenue — Tìm nguồn gốc chính xác ─────────
print("=" * 65)
print("🔍 SALES REVENUE — Reverse Engineer từ transactions")
print("=" * 65)

orders = dfs['orders'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'])
oi = dfs['order_items'].copy()
ship = dfs['shipments'].copy()
pay = dfs['payments'].copy()
sales = dfs['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])

oi['net_rev'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount']

# Thử các filter status khác nhau
for status_filter, label in [
    (['delivered', 'returned', 'shipped', 'paid', 'created'], 'exclude cancelled only'),
    (['delivered', 'returned', 'shipped'],                    'shipped/delivered/returned'),
    (['delivered', 'returned'],                               'delivered + returned'),
    (['delivered'],                                           'delivered only'),
]:
    valid_ids = orders[orders['order_status'].isin(status_filter)]['order_id']
    daily = (oi[oi['order_id'].isin(valid_ids)]
             .merge(orders[['order_id','order_date']], on='order_id')
             .groupby('order_date')['net_rev'].sum()
             .reset_index().rename(columns={'order_date':'Date','net_rev':'rev_oi'}))
    cmp = sales.merge(daily, on='Date', how='left')
    ratio = (cmp['rev_oi'] / cmp['Revenue']).median()
    corr  = cmp['Revenue'].corr(cmp['rev_oi'])
    print(f"  [{label}]")
    print(f"    ratio={ratio:.4f}  corr={corr:.4f}")

# Thử cộng thêm shipping_fee
print()
ship['ship_date_dt'] = pd.to_datetime(ship['ship_date'])
valid_ids_all = orders[orders['order_status'] != 'cancelled']['order_id']

daily_rev = (oi[oi['order_id'].isin(valid_ids_all)]
             .merge(orders[['order_id','order_date']], on='order_id')
             .groupby('order_date')['net_rev'].sum().reset_index()
             .rename(columns={'order_date':'Date'}))

daily_ship = (ship[ship['order_id'].isin(valid_ids_all)]
              .groupby('ship_date_dt')['shipping_fee'].sum().reset_index()
              .rename(columns={'ship_date_dt':'Date'}))

combined = (daily_rev.merge(daily_ship, on='Date', how='left')
            .fillna(0))
combined['total'] = combined['net_rev'] + combined['shipping_fee']
cmp2 = sales.merge(combined, on='Date', how='left')
ratio2 = (cmp2['total'] / cmp2['Revenue']).median()
corr2  = cmp2['Revenue'].corr(cmp2['total'])
print(f"  [exclude cancelled + shipping_fee by ship_date]")
print(f"    ratio={ratio2:.4f}  corr={corr2:.4f}")

# shipping fee theo order_date
daily_ship_by_order = (ship[ship['order_id'].isin(valid_ids_all)]
                       .merge(orders[['order_id','order_date']], on='order_id')
                       .groupby('order_date')['shipping_fee'].sum().reset_index()
                       .rename(columns={'order_date':'Date'}))
combined2 = daily_rev.merge(daily_ship_by_order, on='Date', how='left').fillna(0)
combined2['total'] = combined2['net_rev'] + combined2['shipping_fee']
cmp3 = sales.merge(combined2, on='Date', how='left')
ratio3 = (cmp3['total'] / cmp3['Revenue']).median()
corr3  = cmp3['Revenue'].corr(cmp3['total'])
print(f"  [exclude cancelled + shipping_fee by order_date]")
print(f"    ratio={ratio3:.4f}  corr={corr3:.4f}")

🔍 SALES REVENUE — Reverse Engineer từ transactions
  [exclude cancelled only]
    ratio=0.8842  corr=0.9906
  [shipped/delivered/returned]
    ratio=0.8482  corr=0.9902
  [delivered + returned]
    ratio=0.8245  corr=0.9897
  [delivered only]
    ratio=0.7678  corr=0.9887

  [exclude cancelled + shipping_fee by ship_date]
    ratio=0.8843  corr=0.9906
  [exclude cancelled + shipping_fee by order_date]
    ratio=0.8843  corr=0.9907


In [25]:
# ── CELL I: COGS — Tính ngược từ products × order_items ──────
print("\n" + "=" * 65)
print("💰 COGS — Reverse Engineer")
print("=" * 65)

prod = dfs['products'].copy()
sales = dfs['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])

# Merge order_items với products để lấy cogs per unit
oi_cogs = oi.merge(prod[['product_id','cogs']], on='product_id', how='left')
oi_cogs['total_cogs'] = oi_cogs['cogs'] * oi_cogs['quantity']

# Thử các status filter tương tự
for status_filter, label in [
    (['delivered', 'returned', 'shipped', 'paid', 'created'], 'exclude cancelled only'),
    (['delivered', 'returned', 'shipped'],                    'shipped/delivered/returned'),
    (['delivered'],                                           'delivered only'),
]:
    valid_ids = orders[orders['order_status'].isin(status_filter)]['order_id']
    daily_cogs = (oi_cogs[oi_cogs['order_id'].isin(valid_ids)]
                  .merge(orders[['order_id','order_date']], on='order_id')
                  .groupby('order_date')['total_cogs'].sum()
                  .reset_index().rename(columns={'order_date':'Date'}))
    cmp = sales.merge(daily_cogs, on='Date', how='left')
    ratio = (cmp['total_cogs'] / cmp['COGS']).median()
    corr  = cmp['COGS'].corr(cmp['total_cogs'])
    print(f"  [{label}]  ratio={ratio:.4f}  corr={corr:.4f}")

# Kiểm tra products.cogs có nhiều version không — dùng cogs nào?
print(f"\n  Với versioned products, cogs có nhất quán không?")
dup_prods = prod.groupby(['product_name','size','color']).filter(lambda x: len(x) > 1)
cogs_spread = (dup_prods.groupby(['product_name','size','color'])
               .agg(cogs_min=('cogs','min'), cogs_max=('cogs','max'))
               .assign(cogs_ratio=lambda x: x['cogs_max']/x['cogs_min']))
print(f"  SKU có cogs_max/cogs_min > 2x : {(cogs_spread['cogs_ratio']>2).sum()}")
print(f"  cogs_ratio stats:")
print(cogs_spread['cogs_ratio'].describe().to_string())


💰 COGS — Reverse Engineer
  [exclude cancelled only]  ratio=0.9110  corr=0.9984
  [shipped/delivered/returned]  ratio=0.8778  corr=0.9976
  [delivered only]  ratio=0.8002  corr=0.9959

  Với versioned products, cogs có nhất quán không?
  SKU có cogs_max/cogs_min > 2x : 90
  cogs_ratio stats:
count    186.0000
mean     108.7112
std      207.7675
min        1.0001
25%        1.3133
50%        1.8930
75%      145.1363
max     1432.3745


In [26]:
# ── CELL J: web_traffic — Tính ngược conversion_rate ─────────
print("\n" + "=" * 65)
print("🌐 WEB_TRAFFIC — Verify & Reconstruct conversion_rate")
print("=" * 65)

import os
wt_path = Path('input') / 'web_traffic.csv'
wt = pd.read_csv(wt_path, parse_dates=['date']) if wt_path.exists() else None

if wt is not None:
    print(f"  Columns: {list(wt.columns)}")
    print(f"  Date range: {wt['date'].min()} → {wt['date'].max()}")
    print(f"  Rows: {len(wt):,}")

    # Tính orders per day từ orders table
    orders_per_day = (orders[orders['order_status'] != 'cancelled']
                      .groupby('order_date')['order_id'].count()
                      .reset_index().rename(columns={'order_date':'date','order_id':'n_orders'}))
    
    wt_merged = wt.merge(orders_per_day, on='date', how='left').fillna(0)
    wt_merged['calc_conversion'] = wt_merged['n_orders'] / wt_merged['sessions']

    print(f"\n  Calculated conversion_rate stats:")
    print(wt_merged['calc_conversion'].describe().to_string())

    # Q4: traffic source bounce_rate
    print(f"\n  Q4 — Bounce rate by traffic_source:")
    print(wt.groupby('traffic_source')['bounce_rate'].mean()
          .sort_values().to_string())
else:
    print("  ⚠️ web_traffic.csv not loaded — check INPUT_DIR")


🌐 WEB_TRAFFIC — Verify & Reconstruct conversion_rate
  Columns: ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
  Date range: 2013-01-01 00:00:00 → 2022-12-31 00:00:00
  Rows: 3,652

  Calculated conversion_rate stats:
count   3652.0000
mean       0.0067
std        0.0046
min        0.0003
25%        0.0032
50%        0.0057
75%        0.0086
max        0.0407

  Q4 — Bounce rate by traffic_source:
traffic_source
email_campaign   0.0045
social_media     0.0045
paid_search      0.0045
referral         0.0045
organic_search   0.0045
direct           0.0045


In [27]:
# ── CELL K: Installment anomaly — kỳ 2 là gì? ────────────────
print("\n" + "=" * 65)
print("💳 PAYMENTS — Installment Anomaly Deep Dive")
print("=" * 65)

pay = dfs['payments'].copy()

print("  Avg payment_value by installments:")
print(pay.groupby('installments')['payment_value'].agg(['mean','median','count']).to_string())

# Kỳ 2 — profile đơn hàng
inst2 = pay[pay['installments'] == 2].merge(
    orders[['order_id','order_status','order_date']], on='order_id')
print(f"\n  Installment=2: {len(inst2):,} orders")
print(f"  order_status breakdown:")
print(inst2['order_status'].value_counts().to_string())
print(f"\n  payment_value distribution (installment=2):")
print(inst2['payment_value'].describe().to_string())
print(f"\n  Có phải toàn bộ là cancelled/created không?")
print(f"  Non-delivered rate: {(inst2['order_status'] != 'delivered').mean()*100:.1f}%")

# So sánh với kỳ 1 để thấy sự khác biệt
inst1 = pay[pay['installments'] == 1].merge(
    orders[['order_id','order_status']], on='order_id')
print(f"\n  Installment=1 order_status for comparison:")
print(inst1['order_status'].value_counts(normalize=True).mul(100).round(1).to_string())


💳 PAYMENTS — Installment Anomaly Deep Dive
  Avg payment_value by installments:
                   mean     median   count
installments                              
1            24113.2742 17088.8850  262866
2              708.4737   722.3000    1094
3            24399.6355 17366.7200  218949
6            24446.6544 17451.9650  109910
12           24245.7727 17336.7500   54126

  Installment=2: 1,094 orders
  order_status breakdown:
order_status
delivered    875
cancelled    104
returned      64
shipped       20
paid          16
created       15

  payment_value distribution (installment=2):
count   1094.0000
mean     708.4737
std      128.8520
min      389.7400
25%      627.1450
50%      722.3000
75%      804.3850
max      998.2600

  Có phải toàn bộ là cancelled/created không?
  Non-delivered rate: 20.0%

  Installment=1 order_status for comparison:
order_status
delivered   77.3000
cancelled   11.0000
returned     6.4000
shipped      2.1000
paid         2.1000
created      1.1000


In [28]:
# ── CELL L: Sales seasonality — generator có dùng pattern gì? 
print("\n" + "=" * 65)
print("📅 SALES — Seasonality & Generator Pattern")
print("=" * 65)

sales = dfs['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales['year']  = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month
sales['dow']   = sales['Date'].dt.dayofweek  # 0=Mon
sales['week']  = sales['Date'].dt.isocalendar().week.astype(int)

# Weekend effect
sales['is_weekend'] = sales['dow'].isin([5,6])
print("  Weekend vs Weekday Revenue:")
print(sales.groupby('is_weekend')['Revenue'].agg(['mean','median']).to_string())

# Monthly pattern (average across years)
print(f"\n  Monthly avg Revenue (across all years):")
print(sales.groupby('month')['Revenue'].mean().round(0).to_string())

# YoY growth
print(f"\n  Annual total Revenue:")
print(sales.groupby('year')['Revenue'].sum().round(0).to_string())

# Có missing dates không?
full_range = pd.date_range(sales['Date'].min(), sales['Date'].max(), freq='D')
missing = full_range.difference(sales['Date'])
print(f"\n  Missing dates trong sales.csv: {len(missing)}")
if len(missing) > 0:
    print(f"  Sample: {list(missing[:5])}")


📅 SALES — Seasonality & Generator Pattern
  Weekend vs Weekday Revenue:
                   mean       median
is_weekend                          
False      4405139.6349 3734562.9400
True       3990140.8815 3501407.5100

  Monthly avg Revenue (across all years):
month
1    2591155.0000
2    3480801.0000
3    4928185.0000
4    6532952.0000
5    6575416.0000
6    6427109.0000
7    4659789.0000
8    4441193.0000
9    3797826.0000
10   3302725.0000
11   2611295.0000
12   2524350.0000

  Annual total Revenue:
year
2012    741497748.0000
2013   1657169417.0000
2014   1871845883.0000
2015   1889933827.0000
2016   2104640678.0000
2017   1911164325.0000
2018   1850122456.0000
2019   1136801442.0000
2020   1054512159.0000
2021   1043039820.0000
2022   1169748832.0000

  Missing dates trong sales.csv: 0


In [29]:
# ── CELL M: Inventory — Basic structure & sanity ─────────────
print("=" * 65)
print("📦 INVENTORY — Structure & Logic Check")
print("=" * 65)

inv = pd.read_csv(Path('input') / 'inventory.csv',
                  parse_dates=['snapshot_date'], low_memory=False)

print(f"  Shape: {inv.shape}")
print(f"  Columns: {list(inv.columns)}")
print(f"\n  Date range: {inv['snapshot_date'].min()} → {inv['snapshot_date'].max()}")
print(f"  Unique snapshot_dates: {inv['snapshot_date'].nunique()}")
print(f"  Unique product_ids: {inv['product_id'].nunique()}")
print(f"  Rows per snapshot (should be ~n_products):")
print(inv.groupby(inv['snapshot_date'].dt.to_period('M')).size().describe().to_string())

# Missing values
miss = inv.isnull().sum()
miss = miss[miss > 0]
print(f"\n  Missing values:")
print(miss.to_string() if len(miss) > 0 else "  None")

# Derived columns: year, month có khớp snapshot_date không?
inv['year_check']  = inv['snapshot_date'].dt.year
inv['month_check'] = inv['snapshot_date'].dt.month
year_mismatch  = (inv['year']  != inv['year_check']).sum()
month_mismatch = (inv['month'] != inv['month_check']).sum()
print(f"\n  year/month vs snapshot_date mismatch: {year_mismatch} / {month_mismatch}")

# Numeric sanity
print(f"\n  stock_on_hand — có giá trị âm không: {(inv['stock_on_hand'] < 0).sum()}")
print(f"  units_sold    — có giá trị âm không: {(inv['units_sold']    < 0).sum()}")
print(f"  stockout_days — max: {inv['stockout_days'].max()}, có > 31 không: {(inv['stockout_days'] > 31).sum()}")
print(f"  fill_rate     — ngoài [0,1]: {((inv['fill_rate'] < 0) | (inv['fill_rate'] > 1)).sum()}")
print(f"  sell_through_rate — ngoài [0,1]: {((inv['sell_through_rate'] < 0) | (inv['sell_through_rate'] > 1)).sum()}")

# Flag columns
for flag in ['stockout_flag', 'overstock_flag', 'reorder_flag']:
    vc = inv[flag].value_counts()
    print(f"\n  {flag}: {vc.to_dict()}")

📦 INVENTORY — Structure & Logic Check
  Shape: (60247, 17)
  Columns: ['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate', 'product_name', 'category', 'segment', 'year', 'month']

  Date range: 2012-07-31 00:00:00 → 2022-12-31 00:00:00
  Unique snapshot_dates: 126
  Unique product_ids: 1624
  Rows per snapshot (should be ~n_products):
count   126.0000
mean    478.1508
std      40.5459
min     376.0000
25%     454.2500
50%     481.5000
75%     506.5000
max     562.0000

  Missing values:
  None

  year/month vs snapshot_date mismatch: 0 / 0

  stock_on_hand — có giá trị âm không: 0
  units_sold    — có giá trị âm không: 0
  stockout_days — max: 28, có > 31 không: 0
  fill_rate     — ngoài [0,1]: 0
  sell_through_rate — ngoài [0,1]: 0

  stockout_flag: {1: 40571, 0: 19676}

  overstock_flag: {1: 45942, 0: 14305}

  reorder_flag: {0: 60247}


In [30]:
# ── CELL N: Inventory — Cross-check với order_items ──────────
print("\n" + "=" * 65)
print("📦 INVENTORY — units_sold vs order_items reconcile")
print("=" * 65)

orders = dfs['orders'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'])
oi = dfs['order_items'].copy()

# Tính units_sold per product per month từ order_items
oi_orders = oi.merge(orders[['order_id','order_date','order_status']], on='order_id')
oi_orders = oi_orders[oi_orders['order_status'] != 'cancelled']
oi_orders['year_month'] = oi_orders['order_date'].dt.to_period('M')

sold_from_oi = (oi_orders.groupby(['product_id', 'year_month'])['quantity']
                .sum().reset_index()
                .rename(columns={'quantity': 'units_sold_oi'}))

# Từ inventory
inv['year_month'] = inv['snapshot_date'].dt.to_period('M')
inv_sold = inv[['product_id','year_month','units_sold']].copy()

cmp = inv_sold.merge(sold_from_oi, on=['product_id','year_month'], how='inner')
cmp['diff'] = cmp['units_sold'] - cmp['units_sold_oi']
cmp['ratio'] = cmp['units_sold_oi'] / cmp['units_sold'].replace(0, np.nan)

print(f"  Matched rows: {len(cmp):,}")
print(f"\n  units_sold (inventory) vs quantity (order_items) diff:")
print(cmp['diff'].describe().to_string())
print(f"\n  ratio (oi / inventory) stats:")
print(cmp['ratio'].describe().to_string())
print(f"\n  Exact match (diff=0): {(cmp['diff'] == 0).mean()*100:.1f}%")


📦 INVENTORY — units_sold vs order_items reconcile
  Matched rows: 55,546

  units_sold (inventory) vs quantity (order_items) diff:
count   55546.0000
mean      -35.8449
std        63.2293
min     -1590.0000
25%       -39.0000
50%       -14.0000
75%        -5.0000
max        27.0000

  ratio (oi / inventory) stats:
count   55546.0000
mean        3.4372
std         1.6220
min         0.1111
25%         2.5000
50%         3.1875
75%         4.0000
max        21.0000

  Exact match (diff=0): 3.2%


In [31]:
# ── CELL O: Inventory — Derived metrics verify ───────────────
print("\n" + "=" * 65)
print("📦 INVENTORY — Derived Metrics Logic Check")
print("=" * 65)

# days_of_supply = stock_on_hand / (units_sold / days_in_month)
# Tính số ngày trong tháng
inv['days_in_month'] = inv['snapshot_date'].dt.days_in_month
inv['dos_calc'] = (inv['stock_on_hand'] / 
                   (inv['units_sold'] / inv['days_in_month'])
                   .replace(0, np.nan))

diff_dos = (inv['days_of_supply'] - inv['dos_calc']).abs()
print(f"  days_of_supply vs calculated — mean abs diff: {diff_dos.mean():.4f}")
print(f"  Exact match (<0.01): {(diff_dos < 0.01).mean()*100:.1f}%")

# sell_through_rate = units_sold / (units_sold + stock_on_hand)
inv['str_calc'] = (inv['units_sold'] / 
                   (inv['units_sold'] + inv['stock_on_hand'])
                   .replace(0, np.nan))
diff_str = (inv['sell_through_rate'] - inv['str_calc']).abs()
print(f"\n  sell_through_rate vs calculated — mean abs diff: {diff_str.mean():.4f}")
print(f"  Exact match (<0.01): {(diff_str < 0.01).mean()*100:.1f}%")

# stockout_flag consistency với stockout_days
flag_vs_days = inv.groupby('stockout_flag')['stockout_days'].agg(['mean','min','max'])
print(f"\n  stockout_flag vs stockout_days:")
print(flag_vs_days.to_string())

# Correlation của inventory metrics với sales.Revenue
sales = dfs['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales['year_month'] = sales['Date'].dt.to_period('M')
sales_monthly = sales.groupby('year_month')['Revenue'].sum().reset_index()

inv_monthly = inv.groupby('year_month').agg(
    total_stock=('stock_on_hand','sum'),
    total_sold=('units_sold','sum'),
    avg_fill_rate=('fill_rate','mean'),
    avg_stockout_days=('stockout_days','mean')
).reset_index()

cmp_sal = sales_monthly.merge(inv_monthly, on='year_month', how='inner')
print(f"\n  Correlation với sales.Revenue:")
for col in ['total_stock','total_sold','avg_fill_rate','avg_stockout_days']:
    print(f"    {col:<25}: {cmp_sal['Revenue'].corr(cmp_sal[col]):.4f}")


📦 INVENTORY — Derived Metrics Logic Check
  days_of_supply vs calculated — mean abs diff: 22.4810
  Exact match (<0.01): 23.7%

  sell_through_rate vs calculated — mean abs diff: 0.0000
  Exact match (<0.01): 100.0%

  stockout_flag vs stockout_days:
                mean  min  max
stockout_flag                 
0             0.0000    0    0
1             1.7235    1   28

  Correlation với sales.Revenue:
    total_stock              : -0.1821
    total_sold               : 0.8807
    avg_fill_rate            : -0.4324
    avg_stockout_days        : 0.4325


In [32]:
# ── CELL P: inventory_enhanced — So sánh với inventory ───────
print("\n" + "=" * 65)
print("📦 INVENTORY_ENHANCED — Diff vs inventory.csv")
print("=" * 65)

enh_path = Path('input') / 'inventory_enhanced.csv'
if enh_path.exists():
    enh = pd.read_csv(enh_path, parse_dates=['snapshot_date'], low_memory=False)
    print(f"  Shape: {enh.shape}")
    print(f"  Columns: {list(enh.columns)}")
    
    # So sánh columns
    inv_cols = set(inv.columns) - {'year_check','month_check','days_in_month',
                                    'dos_calc','str_calc','year_month'}
    enh_cols = set(enh.columns)
    print(f"\n  Cols only in inventory    : {inv_cols - enh_cols}")
    print(f"  Cols only in enhanced     : {enh_cols - inv_cols}")
    print(f"  Cols in both              : {len(inv_cols & enh_cols)}")
    
    # Các cột mới trong enhanced có ý nghĩa gì?
    new_cols = enh_cols - inv_cols
    if new_cols:
        print(f"\n  Stats của các cột mới:")
        print(enh[list(new_cols)].describe().to_string())
    
    # Row count match?
    print(f"\n  inventory rows    : {len(inv):,}")
    print(f"  enhanced rows     : {len(enh):,}")
    print(f"  Diff              : {len(enh) - len(inv):,}")
else:
    print("  ⚠️  inventory_enhanced.csv not found")


📦 INVENTORY_ENHANCED — Diff vs inventory.csv
  ⚠️  inventory_enhanced.csv not found


In [33]:
# ── CELL Q: Web traffic — Q4 bounce rate raw values ──────────
print("\n" + "=" * 65)
print("🌐 WEB_TRAFFIC — Q4 Bounce Rate Raw Verification")
print("=" * 65)

wt = pd.read_csv(Path('input') / 'web_traffic.csv', parse_dates=['date'])

# Xem raw values nhiều decimal
pd.set_option('display.float_format', '{:.10f}'.format)
print("  Bounce rate by source (10 decimal places):")
print(wt.groupby('traffic_source')['bounce_rate'].mean().sort_values().to_string())

# Distribution per source
print(f"\n  Bounce rate describe per source:")
print(wt.groupby('traffic_source')['bounce_rate'].describe().to_string())

# Có thực sự identical không?
br_by_source = wt.groupby('traffic_source')['bounce_rate'].mean()
print(f"\n  Tất cả source có bounce_rate bằng nhau: {br_by_source.nunique() == 1}")
print(f"  Min: {br_by_source.min():.10f}")
print(f"  Max: {br_by_source.max():.10f}")

# Web traffic date range vs sales date range  
sales = dfs['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
print(f"\n  sales.csv range  : {sales['Date'].min().date()} → {sales['Date'].max().date()}")
print(f"  web_traffic range: {wt['date'].min().date()} → {wt['date'].max().date()}")
gap_days = (wt['date'].min() - sales['Date'].min()).days
print(f"  Gap (web_traffic starts later by): {gap_days} days")

# Missing dates in web_traffic
full_range = pd.date_range(wt['date'].min(), wt['date'].max(), freq='D')
missing_wt = full_range.difference(wt['date'])
print(f"  Missing dates in web_traffic: {len(missing_wt)}")

pd.set_option('display.float_format', '{:.4f}'.format)


🌐 WEB_TRAFFIC — Q4 Bounce Rate Raw Verification
  Bounce rate by source (10 decimal places):
traffic_source
email_campaign   0.0044584356
social_media     0.0044762816
paid_search      0.0044777679
referral         0.0044988000
organic_search   0.0045042385
direct           0.0045106391

  Bounce rate describe per source:
                         count         mean          std          min          25%          50%          75%          max
traffic_source                                                                                                           
direct          266.0000000000 0.0045106391 0.0007545655 0.0032200000 0.0039400000 0.0044350000 0.0052200000 0.0057800000
email_campaign  505.0000000000 0.0044584356 0.0007333326 0.0032000000 0.0038400000 0.0043900000 0.0051100000 0.0058000000
organic_search 1090.0000000000 0.0045042385 0.0007655564 0.0032000000 0.0038425000 0.0044900000 0.0051900000 0.0058000000
paid_search     784.0000000000 0.0044777679 0.0007597918 0.003200

In [54]:
inv[ (inv['overstock_flag']==1)]

,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month,year_month
3,2016-04-30,3,35,13,11,2,95.5000,0.9333,1,1,0,0.2391,DragonWear MA-03,Casual,All-weather,2016,4,2016-04
4,2016-05-31,3,36,11,10,1,108.0000,0.9667,1,1,0,0.2174,DragonWear MA-03,Casual,All-weather,2016,5,2016-05
5,2016-06-30,3,37,8,7,2,158.6000,0.9333,1,1,0,0.1591,DragonWear MA-03,Casual,All-weather,2016,6,2016-06
6,2016-07-31,3,39,11,9,2,130.0000,0.9333,1,1,0,0.1875,DragonWear MA-03,Casual,All-weather,2016,7,2016-07
7,2016-08-31,3,39,4,4,0,292.5000,1.0000,0,1,0,0.0930,DragonWear MA-03,Casual,All-weather,2016,8,2016-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60242,2022-08-31,2412,251,25,21,2,358.6000,0.9333,1,1,0,0.0772,VietMotion YY-21,GenZ,Trendy,2022,8,2022-08
60243,2022-09-30,2412,251,3,3,2,2510.0000,0.9333,1,1,0,0.0118,VietMotion YY-21,GenZ,Trendy,2022,9,2022-09
60244,2022-10-31,2412,251,3,3,0,2510.0000,1.0000,0,1,0,0.0118,VietMotion YY-21,GenZ,Trendy,2022,10,2022-10
60245,2022-11-30,2412,251,3,3,0,2510.0000,1.0000,0,1,0,0.0118,VietMotion YY-21,GenZ,Trendy,2022,11,2022-11


In [52]:
# ── CELL R: Inventory flags — deep dive ──────────────────────
print("=" * 65)
print("🚩 INVENTORY FLAGS — Co-occurrence & Temporal Pattern")
print("=" * 65)

inv = pd.read_csv(Path('input') / 'inventory.csv',
                  parse_dates=['snapshot_date'], low_memory=False)
inv['year_month'] = inv['snapshot_date'].dt.to_period('M')
inv['year'] = inv['snapshot_date'].dt.year

# Co-occurrence
both  = ((inv['overstock_flag'] == 1) & (inv['stockout_flag'] == 1)).sum()
over_only = ((inv['overstock_flag'] == 1) & (inv['stockout_flag'] == 0)).sum()
stock_only = ((inv['overstock_flag'] == 0) & (inv['stockout_flag'] == 1)).sum()
neither = ((inv['overstock_flag'] == 0) & (inv['stockout_flag'] == 0)).sum()
print(f"\n  Flag co-occurrence:")
print(f"    Both flags = 1        : {both:,}  ({both/len(inv)*100:.1f}%)")
print(f"    Overstock only        : {over_only:,}  ({over_only/len(inv)*100:.1f}%)")
print(f"    Stockout only         : {stock_only:,}  ({stock_only/len(inv)*100:.1f}%)")
print(f"    Neither               : {neither:,}  ({neither/len(inv)*100:.1f}%)")

# Overstock_flag trend theo năm — có thay đổi không?
print(f"\n  overstock_flag rate theo năm:")
print(inv.groupby('year')['overstock_flag'].mean().mul(100).round(1).to_string())

print(f"\n  stockout_flag rate theo năm:")
print(inv.groupby('year')['stockout_flag'].mean().mul(100).round(1).to_string())

# Theo category
prod = dfs['products'].copy()
inv_prod = inv.merge(prod[['product_id','category','segment']].drop_duplicates('product_id'),
                     on='product_id', how='left')
print(f"\n  Flag rates theo category:")
flag_by_cat = inv_prod.groupby('category')[['overstock_flag','stockout_flag']].mean().mul(100).round(1)
print(flag_by_cat.to_string())

# overstock_flag vs stock_on_hand / units_sold ratio
inv['stock_ratio'] = inv['stock_on_hand'] / inv['units_sold'].replace(0, np.nan)
print(f"\n  stock_on_hand / units_sold ratio by overstock_flag:")
print(inv.groupby('overstock_flag')['stock_ratio'].describe().to_string())

# Correlation của flags với monthly revenue
sales = dfs['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales['year_month'] = sales['Date'].dt.to_period('M')
sales_monthly = sales.groupby('year_month')['Revenue'].sum().reset_index()

inv_monthly_flags = inv.groupby('year_month').agg(
    overstock_rate=('overstock_flag','mean'),
    stockout_rate=('stockout_flag','mean'),
    both_rate=('overstock_flag', lambda x: 
               ((x == 1) & (inv.loc[x.index,'stockout_flag'] == 1)).mean())
).reset_index()

cmp = sales_monthly.merge(inv_monthly_flags, on='year_month', how='inner')
print(f"\n  Correlation với monthly Revenue:")
for col in ['overstock_rate','stockout_rate','both_rate']:
    print(f"    {col:<20}: {cmp['Revenue'].corr(cmp[col]):.4f}")

🚩 INVENTORY FLAGS — Co-occurrence & Temporal Pattern

  Flag co-occurrence:
    Both flags = 1        : 30,495  (50.6%)
    Overstock only        : 15,447  (25.6%)
    Stockout only         : 10,076  (16.7%)
    Neither               : 4,229  (7.0%)

  overstock_flag rate theo năm:
year
2012   56.9000
2013   66.1000
2014   74.0000
2015   77.9000
2016   77.8000
2017   78.7000
2018   79.3000
2019   81.8000
2020   82.0000
2021   76.4000
2022   76.3000

  stockout_flag rate theo năm:
year
2012   68.2000
2013   67.8000
2014   67.6000
2015   67.5000
2016   67.9000
2017   67.8000
2018   66.5000
2019   66.9000
2020   66.7000
2021   67.5000
2022   66.7000

  Flag rates theo category:


KeyError: 'category'

In [51]:
inv.columns

Index(['snapshot_date', 'product_id', 'stock_on_hand', 'units_received',
       'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate',
       'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate',
       'product_name', 'category', 'segment', 'year', 'month', 'year_month'],
      dtype='object')